In [34]:
import carla
import numpy as np
import logging
import pygame
import cv2
import random


In [ ]:
CARLA_LOCALHOST = "localhost"
CARLA_PORT = 2000
CARLA_TIMEOUT = 40.0

logging.basicConfig(
    level = logging.INFO,
    format = "%(asctime)s [%(levelname)s] %(name)s: %(message)s"
)

random.seed(42)

logger = logging.getLogger(__name__)
IMAGE_Z = 3
IMAGE_X = 0
CAMERA_INIT_TRANS = carla.Transform(carla.Location(z = IMAGE_Z, x = IMAGE_X))

In [3]:
def connect_to_carla() -> tuple[carla.Client, carla.World]:

    client = carla.Client(CARLA_LOCALHOST, CARLA_PORT)
    client.set_timeout(CARLA_TIMEOUT)
    world = client.get_world()

    if not world:
        raise RuntimeError(
            "Carla was not able to return a valid world."
        )

    logger.info(
        "Connection to client established."
    )
    return client, world

In [29]:
def load_map(world: carla.World, client: carla.Client, map: str = "Town05") -> None:

    if world.get_map().name.split("/")[-1] == map:
        print("Assigned map is already loaded")
    else:
        try:
            world = client.load_world(map)
        except:
            world = client.load_world("Town01")
        finally:
            logger.info(
                f"Map: {map} is loaded."
            )
            

In [21]:
def set_spectator(world: carla.World) -> None:

    spectator = world.get_spectator()
    spectator.set_transform(
        carla.Transform(
            carla.Location(
                x = 0,
                y = 0,
                z = 200
            ),
            carla.Rotation(
                pitch = -90,
                yaw = 0,
                roll = 0
            )
        )
    )

In [42]:
def load_actors(world: carla.World) -> None:
    vehicle_bp = world.get_blueprint_library().filter("vehicle.*")
    spawn_points = world.get_map().get_spawn_points()
    vehicles = []

    for i in range(0, 10, 1):
        spawn_point = random.choice(spawn_points)
        blueprint = random.choice(vehicle_bp)

        vehicle = world.try_spawn_actor(
            blueprint,
            spawn_point
        )

        if vehicle is not None:
            vehicle.set_autopilot(True)
            vehicles.append(vehicle)

In [50]:
def ego_vehicle(world: carla.World) -> carla.Vehicle:

    vehicle_bp = world.get_blueprint_library().filter("vehicle.*")
    spawn_points = world.get_map().get_spawn_points()
    spawn_point = random.choice(spawn_points)
    if not vehicle_bp:
        raise RuntimeError(
            "No valid blueprint return to cybertruck."
        )

    vehicle = world.try_spawn_actor(
        vehicle_bp[0],
        spawn_point
    )

    return vehicle

In [ ]:
def main():
    client, world = connect_to_carla()
    actors = world.get_actors().filter("vehicle.*")
    for vehicle in actors:
        vehicle.destroy()
    load_map(world, client, "Town05")
    set_spectator(world)
    load_actors(world)
    vehicle = ego_vehicle(world)

    

In [ ]:
if __name__ == "__main__":
    main()

2026-11-13 23:34:07,686 [INFO] __main__: Connection to client established.


Assigned map is already loaded


: 